# Customer Feedback Sentiment Analysis - Cloud Training Workflow
### MCA Combined Practical Project

This Google Colab notebook provides an alternative cloud-based training environment.
Use this notebook if your local Windows machine enforces **Smart App Control (SAC)** or **Application Control** policies that restrict local C-extension execution.

**Pipeline Overview:**
1. Setup environment and install dependencies
2. Download or upload Amazon Fine Food Reviews (`Reviews.csv`)
3. Clean and preprocess review text
4. Balanced sampling across 3 sentiment categories (Positive, Neutral, Negative)
5. TF-IDF feature extraction (15,000 max features, n-grams 1-2)
6. Logistic Regression training with balanced class weighting
7. Unseen test set evaluation & confusion matrix generation
8. Download trained model artifacts for local Streamlit deployment

In [ ]:
# Step 1: Install required packages in Google Colab
!pip install -q scikit-learn pandas numpy matplotlib joblib

In [ ]:
# Step 2: Download or upload Reviews.csv
# You can upload Reviews.csv directly using the files pane or via Kaggle API:
# !kaggle datasets download -d snap/amazon-fine-food-reviews
# !unzip amazon-fine-food-reviews.zip

import os
import re
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

os.makedirs('models', exist_ok=True)
print('Environment initialized successfully!')

In [ ]:
# Step 3: Load and clean dataset
DATA_PATH = 'Reviews.csv'  # update path if located in a subfolder

df = pd.read_csv(DATA_PATH, usecols=['Score', 'Text'])
df.dropna(subset=['Score', 'Text'], inplace=True)
df['Text'] = df['Text'].astype(str).str.strip()
df = df[df['Text'] != '']
df.drop_duplicates(subset=['Text'], inplace=True)

def map_sentiment(score):
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['Sentiment'] = df['Score'].apply(map_sentiment)

# Balanced sampling: up to 20,000 reviews per sentiment class
SAMPLE_SIZE = 20000
df = (
    df.groupby('Sentiment', group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), SAMPLE_SIZE), random_state=42))
    .reset_index(drop=True)
)
print('Dataset distribution:', df['Sentiment'].value_counts().to_dict())

In [ ]:
# Step 4: Text Preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)           # Strip HTML
    text = re.sub(r'http\S+|www\S+', ' ', text)  # Strip URLs
    text = re.sub(r'[^a-z\s]', ' ', text)        # Keep alphabetic characters
    text = re.sub(r'\s+', ' ', text).strip()      # Normalize spaces
    return text

df['Cleaned_Text'] = df['Text'].apply(clean_text)
df = df[df['Cleaned_Text'] != '']

X = df['Cleaned_Text'].tolist()
y = df['Sentiment'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training: {len(X_train)} samples | Testing: {len(X_test)} samples')

In [ ]:
# Step 5: TF-IDF Vectorization and Logistic Regression Training
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train_tfidf, y_train)
print('Model training complete!')

In [ ]:
# Step 6: Evaluation and Artifact Export
y_pred = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
labels = ['Negative', 'Neutral', 'Positive']
report_dict = classification_report(y_test, y_pred, labels=labels, output_dict=True, zero_division=0)
cm = confusion_matrix(y_test, y_pred, labels=labels)

print(f'Test Accuracy: {acc * 100:.2f}%')
print(classification_report(y_test, y_pred, labels=labels, zero_division=0))

# Save artifacts
joblib.dump(model, 'models/sentiment_model.pkl')
joblib.dump(vectorizer, 'models/tfidf_vectorizer.pkl')

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues', values_format='d', ax=ax)
plt.title('Customer Review Sentiment Confusion Matrix')
plt.tight_layout()
plt.savefig('models/confusion_matrix.png', dpi=300)
plt.close()

eval_data = {
    'model_name': 'Logistic Regression with TF-IDF',
    'dataset': 'Amazon Fine Food Reviews',
    'total_samples': len(X),
    'training_samples': len(X_train),
    'testing_samples': len(X_test),
    'accuracy': round(float(acc), 4),
    'precision_macro': round(float(report_dict['macro avg']['precision']), 4),
    'recall_macro': round(float(report_dict['macro avg']['recall']), 4),
    'f1_score_macro': round(float(report_dict['macro avg']['f1-score']), 4),
    'class_labels': labels,
    'classification_report': report_dict,
    'confusion_matrix': cm.tolist(),
    'class_distribution': df['Sentiment'].value_counts().to_dict()
}

with open('models/evaluation_results.json', 'w') as f:
    json.dump(eval_data, f, indent=4)

print('All 4 artifacts saved into models/ folder!')

# Package artifacts for download
!zip -r trained_models.zip models/
from google.colab import files
files.download('trained_models.zip')